In [1]:
from rdflib import Graph, plugin, URIRef
from rdflib.serializer import Serializer

import json
import requests

In [2]:
url = requests.get("https://p-lod.umasscreate.net/api/items?item_set_id=2&per_page=20")
jsonitems = url.text

url = requests.get("https://p-lod.umasscreate.net/api/items?item_set_id=1")
jsonclasses = url.text


In [3]:
g_classes = Graph().parse(data=jsonclasses, format='json-ld')

In [4]:
g_classes_use = g_classes
g_classes_use.parse(data=jsonitems, format='json-ld')

<Graph identifier=Nff6423499c604d78b817da0a3c562677 (<class 'rdflib.graph.Graph'>)>

In [5]:
# print(g_classes_use.serialize(format='turtle').decode('utf-8'))

In [6]:
rdf_construct  = g_classes_use.query("""
CONSTRUCT { ?new_s a ?type ; ?p_local ?o_local .}
WHERE {?s <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/2> . 

       ?s a ?type_as_item .
       ?type_as_item <http://omeka.org/s/vocabs/o#item_set> <https://p-lod.umasscreate.net/api/item_sets/1> .
       ?type_as_item <http://purl.org/dc/terms/identifier> ?type_as_label .
       BIND(IRI(concat("https://p-lod.umasscreate.net/p-lod/id/",?type_as_label)) as ?type)
       
       ?s <http://purl.org/dc/terms/identifier> ?item_identifier .
       BIND(IRI(CONCAT("https://p-lod.umasscreate.net/p-lod/id/",?item_identifier)) as ?new_s)
       
       ?s ?p_local ?o_local .
       FILTER regex(str(?p_local), "http://p-lod.umasscreate.net/vocabulary#") . 
       
       #?o_local <http://purl.org/dc/terms/identifier> ?object_identifier .
       #BIND(IRI(CONCAT("https://p-lod.umasscreate.net/p-lod/id/",?object_identifier)) as ?new_o)
       }
       
""")


In [9]:
new_g = Graph()
new_g.namespace_manager.bind('p-lod-vocab', URIRef('http://p-lod.umasscreate.net/vocabulary#'))
new_g.namespace_manager.bind('p-lod-items', URIRef('https://p-lod.umasscreate.net/p-lod/id/'))
for triple in rdf_construct:
    if "https://p-lod.umasscreate.net/api/items/" in triple[2]:
        sub_item_json = requests.get(triple[2]).text
        new_uri = URIRef("https://p-lod.umasscreate.net/p-lod/id/" + json.loads(sub_item_json)['dcterms:identifier'][0]['@value'])
        new_g.add((triple[0],triple[1],new_uri))
    else:
        new_g.add(triple)

ttl = new_g.serialize(format="turtle").decode('utf-8')

In [11]:
ttl_file = open("p-lod.ttl", "w")
ttl_file.write(ttl)
ttl_file.close()